# Adapter
  
Dieses Notebook trainiert einen Adapter auf den bestehenden CLIP Embeddings
  


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from scipy.spatial import cKDTree
import geopandas as gpd




def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())

PROJECT_ROOT = find_project_root()
METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
ADAPTER = CFG["vpr"].get("adapter", "none")
METHOD_DIR = EMBEDDING_DIR / f"{METHOD}"
POSITIVE_PATH = PROJECT_ROOT / "data" / "processed" / "positive_candidates.csv"

SEED = int(CFG["vpr"].get("split_seed", 42))
ADAPTER_CFG = CFG["vpr"]["adapter_training"]
BATCH_SIZE = ADAPTER_CFG["batch_size"]
EPOCHS = ADAPTER_CFG["epochs"]  
LEARNING_RATE = ADAPTER_CFG["learning_rate"]
MARGIN = ADAPTER_CFG["margin"]
NEGATIVE_RADIUS_M = CFG["vpr"]["negative_radius_m"]
POSITIVE_RADIUS_M = CFG["vpr"]["positive_radius_m"]
HARD_NEGATIVE_PROBABILITY = ADAPTER_CFG["hard_negative_probability"]


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.adapter import LinearAdapter

if ADAPTER == "none" or ADAPTER == "None":
    EMBEDDING_NAME = METHOD
else:
    EMBEDDING_NAME = f"{METHOD}_{ADAPTER}"


if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"


MODEL_DIR = PROJECT_ROOT / "models" / "adapters"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_PATH = MODEL_DIR / f"{METHOD}_linear.pt"

positive_candidates = pd.read_csv(POSITIVE_PATH)


print(positive_candidates["distance_m"].describe())
print()
print(positive_candidates["distance_m"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Project:             {PROJECT_ROOT}")
print(f"Method:              {METHOD}")
print(f"Model:               {MODEL_ID}")
print(f"Device:              {DEVICE}")
print(f"Batch size:          {BATCH_SIZE}")
print(f"Epochs:              {EPOCHS}")
print(f"Learning rate:       {LEARNING_RATE}")
print(f"Margin:              {MARGIN}")
print(f"Positive radius:     {POSITIVE_RADIUS_M} m")
print(f"Negative radius:     {NEGATIVE_RADIUS_M} m")
print(f"Adapter output:      {ADAPTER_PATH}")


# CLIP Embeddings laden
  
Baseline CLIP Embeddings kein trainierete Adapter

In [ ]:


EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD

embedding_path = EMBEDDING_DIR / f"{METHOD}_embeddings.npy"

metadata_path = EMBEDDING_DIR / f"{METHOD}_metadata.parquet"


embeddings = np.load(embedding_path)

embedding_metadata = pd.read_parquet(metadata_path)



if len(embeddings) != len(embedding_metadata):
    raise ValueError(
        f"Embedding/Metadata-Mismatch: "
        f"{len(embeddings)} Embeddings vs. "
        f"{len(embedding_metadata)} Metadata-Zeilen"
    )


print(f"Embeddings: {embeddings.shape}")

print(f"Metadata:   {embedding_metadata.shape}")


In [ ]:
import random

# Nur der train-Split wird zum Trainieren angefasst.
train_mask_all = (embedding_metadata["split"] == "train").to_numpy()
train_metadata_all = embedding_metadata[train_mask_all].reset_index(drop=True)

# Aufteilen auf SEQUENZ-Ebene, nicht auf Bildebene.
rng = random.Random(SPLIT_SEED)
train_seqs = sorted(train_metadata_all["sequence_id"].astype(str).unique())
rng.shuffle(train_seqs)

n_val = max(1, int(0.1 * len(train_seqs)))
val_seqs = set(train_seqs[:n_val])
fit_seqs = set(train_seqs[n_val:])

seq_col = embedding_metadata["sequence_id"].astype(str)
fit_mask = train_mask_all & seq_col.isin(fit_seqs).to_numpy()
val_mask = train_mask_all & seq_col.isin(val_seqs).to_numpy()

fit_embeddings = embeddings[fit_mask]
val_embeddings = embeddings[val_mask]
fit_metadata = embedding_metadata[fit_mask].reset_index(drop=True)
val_metadata = embedding_metadata[val_mask].reset_index(drop=True)

print(f"Train-Sequenzen gesamt: {len(train_seqs):,}")
print(f"  fit: {len(fit_seqs):,} Sequenzen / {fit_mask.sum():,} Bilder")
print(f"  val: {len(val_seqs):,} Sequenzen / {val_mask.sum():,} Bilder")
assert not (fit_seqs & val_seqs), "Sequenz-Leakage zwischen fit und val!"


# Train Mask und Embeddings erstellen

In [ ]:

train_mask = embedding_metadata["split"] == "train"
train_embeddings = embeddings[train_mask]
train_metadata = embedding_metadata[train_mask].reset_index(drop=True)


print(f"Database: {train_embeddings.shape}")


# Positive Candidate Laden

In [ ]:
# ------------------------------------------------------------
# Positive Candidates
# ------------------------------------------------------------

POSITIVE_PATH = PROJECT_ROOT / "data" / "processed" / "positive_candidates.csv"


positive_candidates = pd.read_csv(POSITIVE_PATH)


print(f"Positive candidate rows: {len(positive_candidates):,}")

print(positive_candidates["distance_m"].describe())


# IDs in Embedding Indizes umwandeln

In [ ]:


train_id_to_index = {image_id: index for index, image_id in enumerate(train_metadata["image_id"])}

positive_pairs = []

for row in positive_candidates.itertuples(index = False):
    train_id = row.train_image_id
  

    if train_id not in train_id_to_index:
        continue

    train_index = train_id_to_index[train_id]

    positive_pairs.append((train_index, row.distance_m))


print(f"Positive Pairs:     {len(positive_pairs)}")

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6_371_000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    diff_lat = lat2 - lat1
    diff_lon = lon2 - lon1

    a = (
        np.sin(diff_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(diff_lon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius * c




In [ ]:
def to_metric_xy(lat, lon, crs=None):
    """Lat/Lon -> Meter. Gibt das CRS zurueck, damit alle dasselbe benutzen."""
    gs = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326")
    if crs is None:
        crs = gs.estimate_utm_crs()
    gs = gs.to_crs(crs)
    return np.c_[gs.x.to_numpy(), gs.y.to_numpy()], crs


In [ ]:
class TripletDataset(Dataset):
    def __init__(
        self,
        train_embeddings,
        positive_pairs,
        train_metadata,
        positive_radius_m,
        negative_radius_m,
        hard_negative_probability=0.75,
    ):
        self.train_embeddings = train_embeddings
        self.positive_pairs = positive_pairs

        self.train_metadata = train_metadata 

        self.positive_radius_m = positive_radius_m
        self.negative_radius_m = negative_radius_m
        self.hard_negative_probability = hard_negative_probability

        # Positive Database-Indizes pro Train
        self.positive_by_train = {}

        for train_index, database_index, distance_m in positive_pairs:
            self.positive_by_train.setdefault(
                train_index,
                set(),
            ).add(database_index)

        # --- einmalig statt 219.690-mal ------------------------------------
        db_xy, crs = to_metric_xy(
            database_metadata["lat"].to_numpy(),
            database_metadata["lon"].to_numpy(),
        )
        q_xy, _ = to_metric_xy(
            query_metadata["lat"].to_numpy(),
            query_metadata["lon"].to_numpy(),
            crs=crs,  # gleiches CRS erzwingen
        )

        self.n_database = len(db_xy)
        tree = cKDTree(db_xy)

        # Nur Queries, die ueberhaupt in positive_pairs vorkommen.
        needed = sorted(self.positive_by_query.keys())

        # query_ball_point akzeptiert ein Array -> ein Aufruf statt 53.000.
        near_neg = tree.query_ball_point(q_xy[needed], r=negative_radius_m)
        near_pos = tree.query_ball_point(q_xy[needed], r=positive_radius_m)

        self.hard_by_query = {}  # 10-25 m, ohne Positives
        self.blocked_by_query = {}  # alles <= 25 m, darf kein Easy Negative sein

        for slot, qi in enumerate(needed):
            neg_set = set(near_neg[slot])
            pos_set = set(near_pos[slot]) | self.positive_by_query.get(qi, set())
            hard = neg_set - pos_set
            self.hard_by_query[qi] = np.fromiter(hard, dtype=np.int64, count=len(hard))
            self.blocked_by_query[qi] = neg_set | pos_set

        print(f"Nachbarschaften vorberechnet fuer {len(needed):,} Queries")

    def __len__(self):
        return len(self.positive_pairs)


    def _sample_negative(self, query_index):
    hard = self.hard_by_query.get(query_index)
    blocked = self.blocked_by_query.get(query_index, frozenset())

    if hard is not None and len(hard) and np.random.random() < self.hard_negative_probability:
        return int(hard[np.random.randint(len(hard))])

    # Easy Negative per Rejection Sampling: bei ~61 gesperrten von 48.321
    # Kandidaten trifft der erste Wurf zu 99,87 % daneben-ins-Richtige.
    for _ in range(32):
        candidate = np.random.randint(self.n_database)
        if candidate not in blocked:
            return int(candidate)

    if hard is not None and len(hard):
        return int(hard[np.random.randint(len(hard))])

    raise RuntimeError(f"Keine negativen Kandidaten für Query {query_index}.")

    def __getitem__(self, index):

        train_index, positive_index, _ = self.positive_pairs[index]

        # Anchor
        anchor = self.train_embeddings[train_index]

        # Positive
        positive = self.database_embeddings[positive_index]

        # Negative
        negative_index = self._sample_negative(train_index)
        negative = self.database_embeddings[negative_index]

        return (
            torch.from_numpy(anchor).float(),
            torch.from_numpy(positive).float(),
            torch.from_numpy(negative).float(),
        )


In [ ]:
# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

HARD_NEGATIVE_PROBABILITY = ADAPTER_CFG["hard_negative_probability"]

dataset = TripletDataset(
    train_embeddings=train_embeddings,
    positive_pairs=positive_pairs,
    train_metadata=train_metadata,
    positive_radius_m=POSITIVE_RADIUS_M,
    negative_radius_m=NEGATIVE_RADIUS_M,
    hard_negative_probability=HARD_NEGATIVE_PROBABILITY,
)


print(f"Dataset size: {len(dataset):,}")

# ------------------------------------------------------------
# Negative Sampling überprüfen
# ------------------------------------------------------------

train_index, positive_index, positive_distance = positive_pairs[0]

train = train_metadata.iloc[train_index]

distances = haversine_distance(
    train["lat"],
    train["lon"],
    train_metadata["lat"].to_numpy(),
    train_metadata["lon"].to_numpy(),
)

hard_negatives = np.where(
    (distances > POSITIVE_RADIUS_M) & (distances <= NEGATIVE_RADIUS_M)
)[0]

easy_negatives = np.where(distances > NEGATIVE_RADIUS_M)[0]

print(f"Positive Pairs:   {len(positive_pairs):,}")
print(f"Hard negatives:   {len(hard_negatives):,}")
print(f"Easy negatives:   {len(easy_negatives):,}")


# ------------------------------------------------------------
# Einzelnes Sample
# ------------------------------------------------------------

anchor, positive, negative = dataset[0]


print("Anchor:  ", anchor.shape)

print("Positive:", positive.shape)

print("Negative:", negative.shape)

# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)


anchor, positive, negative = next(
    iter(train_loader)
)


print(
    "Anchor:  ",
    anchor.shape
)

print(
    "Positive:",
    positive.shape
)

print(
    "Negative:",
    negative.shape
)

# Adapter erstellen

In [ ]:

# ------------------------------------------------------------
# Adapter
# ------------------------------------------------------------

EMBEDDING_DIM = embeddings.shape[1]

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

loss_fn = nn.TripletMarginWithDistanceLoss(
    distance_function=lambda x, y: (
        1
        - F.cosine_similarity(
            x,
            y,
            dim=-1,
        )
    ),
    margin=MARGIN,
)
adapter = LinearAdapter(embedding_dim=EMBEDDING_DIM).to(DEVICE)


print(adapter)

# Optimizer

In [ ]:
# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    adapter.parameters(),
    lr=LEARNING_RATE,
)


# Sanity Check vor dem Training

In [ ]:
# ------------------------------------------------------------
# Vorher: Similarity Sanity Check
# ------------------------------------------------------------

adapter.eval()


with torch.inference_mode():
    anchor, positive, negative = next(iter(train_loader))

    anchor = anchor.to(DEVICE)
    positive = positive.to(DEVICE)
    negative = negative.to(DEVICE)

    anchor_out = adapter(anchor)
    positive_out = adapter(positive)
    negative_out = adapter(negative)

    positive_similarity = (
        F.cosine_similarity(
            anchor_out,
            positive_out,
            dim=-1,
        )
        .mean()
        .item()
    )

    negative_similarity = (
        F.cosine_similarity(
            anchor_out,
            negative_out,
            dim=-1,
        )
        .mean()
        .item()
    )


print("Before training:")

print(f"Positive similarity: {positive_similarity:.4f}")

print(f"Negative similarity: {negative_similarity:.4f}")


# Training
  
Du solltest jetzt sehen, wie der Loss im Groben sinkt, wobei er nicht zwingend monoton sinken muss.

In [ ]:
# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

adapter.train()


for epoch in range(EPOCHS):
    total_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )

    for anchor, positive, negative in progress:
        anchor = anchor.to(DEVICE)
        positive = positive.to(DEVICE)
        negative = negative.to(DEVICE)

        optimizer.zero_grad()

        anchor_out = adapter(anchor)
        positive_out = adapter(positive)
        negative_out = adapter(negative)

        loss = loss_fn(
            anchor_out,
            positive_out,
            negative_out,
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}: mean loss = {mean_loss:.4f}")


# Similarity Check nach dem Training
  
                    vorher       nachher

positive similarity   0.XX         ↑
negative similarity   0.XX         ↓

In [ ]:
# ------------------------------------------------------------
# Nachher: Similarity Sanity Check
# ------------------------------------------------------------

adapter.eval()


with torch.inference_mode():
    anchor, positive, negative = next(iter(train_loader))

    anchor = anchor.to(DEVICE)
    positive = positive.to(DEVICE)
    negative = negative.to(DEVICE)

    anchor_out = adapter(anchor)
    positive_out = adapter(positive)
    negative_out = adapter(negative)

    positive_similarity_after = (
        F.cosine_similarity(
            anchor_out,
            positive_out,
            dim=-1,
        )
        .mean()
        .item()
    )

    negative_similarity_after = (
        F.cosine_similarity(
            anchor_out,
            negative_out,
            dim=-1,
        )
        .mean()
        .item()
    )


print("After training:")

print(f"Positive similarity: {positive_similarity_after:.4f}")

print(f"Negative similarity: {negative_similarity_after:.4f}")


# Adapter abspeichern

In [ ]:
# ------------------------------------------------------------
# Adapter speichern
# ------------------------------------------------------------

torch.save(
    adapter.state_dict(),
    ADAPTER_PATH,
)


print("Adapter gespeichert unter:")

print(ADAPTER_PATH)
